# NYC Taxi — Exploratory Data Analysis

Quick EDA over the sample dataset. Run `python data/generate_sample.py` first if
`data/sample/` is empty. This notebook uses pandas/matplotlib for fast, local plots;
the production pipeline in `src/` uses Spark.


In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
df = pd.read_csv('../data/sample/yellow_tripdata_sample.csv',
                 parse_dates=['tpep_pickup_datetime','tpep_dropoff_datetime'])
zones = pd.read_csv('../data/sample/taxi_zone_lookup.csv')
print(df.shape); df.head()


## 1. Cleaning & target
Compute duration and drop physically impossible rows (mirrors `src/transformations.clean_trips`).


In [ ]:
df['duration_min'] = (df['tpep_dropoff_datetime'] - df['tpep_pickup_datetime']).dt.total_seconds()/60
before = len(df)
df = df[(df.trip_distance>0)&(df.fare_amount>=0)&(df.duration_min>0)&(df.duration_min<180)&(df.passenger_count>0)]
print(f'{before:,} -> {len(df):,} rows after cleaning')
df[['trip_distance','duration_min','fare_amount']].describe()


## 2. Trip duration by hour of day
The congestion signal the model exploits.


In [ ]:
df['hour'] = df.tpep_pickup_datetime.dt.hour
by_hour = df.groupby('hour')['duration_min'].mean()
by_hour.plot(kind='bar', figsize=(10,4), title='Avg trip duration by pickup hour')
plt.ylabel('minutes'); plt.tight_layout(); plt.show()


## 3. Distance vs. duration
Distance is the dominant driver; colour shows the rush-hour spread.


In [ ]:
rush = df.hour.between(7,10) | df.hour.between(16,19)
plt.figure(figsize=(7,5))
plt.scatter(df.trip_distance[~rush], df.duration_min[~rush], s=3, alpha=.2, label='off-peak')
plt.scatter(df.trip_distance[rush],  df.duration_min[rush],  s=3, alpha=.2, label='rush')
plt.xlabel('distance (mi)'); plt.ylabel('duration (min)'); plt.legend(); plt.show()


## 4. Demand hotspots
Busiest pickup zones after joining the zone lookup.


In [ ]:
j = df.merge(zones, left_on='PULocationID', right_on='LocationID', how='left')
j.groupby(['Borough','Zone']).size().sort_values(ascending=False).head(10)
